In [1]:
# 1. Install PyTorch Geometric for the GPU environment
!pip install torch_geometric
!pip install optional_dependencies torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.3.0+cu121.html

# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.loader import DataLoader
from torch_geometric.data import Dataset
from torch_geometric.nn import GATConv, global_mean_pool

# 3. Update Directory Configurations
base_dir = '/content/drive/MyDrive/ASD_GNN_Research2'
graphs_dir = os.path.join(base_dir, 'graphs')
checkpoint_dir = os.path.join(base_dir, 'models', 'Checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)

# 4. Custom PyTorch Geometric Dataset Loader
class ABIDEGraphDataset(Dataset):
    def __init__(self, folder_path):
        super().__init__()
        self.folder_path = folder_path
        self.file_names = sorted([f for f in os.listdir(folder_path) if f.endswith('.pt')])

    def len(self):
        return len(self.file_names)

    def get(self, idx):
        file_path = os.path.join(self.folder_path, self.file_names[idx])
        return torch.load(file_path, weights_only=False)

# Initialize split datasets
train_dataset = ABIDEGraphDataset(os.path.join(graphs_dir, 'train'))
val_dataset = ABIDEGraphDataset(os.path.join(graphs_dir, 'val'))
test_dataset = ABIDEGraphDataset(os.path.join(graphs_dir, 'test'))

# Instantiate DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"DataLoaders Ready. Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)}")

# 5. Model Architecture Options
class GATClassifier(torch.nn.Module):
    def __init__(self, num_node_features, hidden_channels, num_classes, heads=4):
        super(GATClassifier, self).__init__()

        self.conv1 = GATConv(num_node_features, hidden_channels, heads=heads, concat=True)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=1, concat=False)

        self.lin1 = Linear(hidden_channels, hidden_channels // 2)
        self.lin2 = Linear(hidden_channels // 2, num_classes)

    def forward(self, x, edge_index, batch, edge_attr=None):
        x = self.conv1(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=0.4, training=self.training)

        x = self.conv2(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)

        x = global_mean_pool(x, batch)

        x = self.lin1(x)
        x = F.elu(x)
        x = F.dropout(x, p=0.4, training=self.training)
        x = self.lin2(x)

        return x

# 6. Define Training Framework
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using execution device: {device}")

# ---> CRITICAL UPDATE: num_node_features is now 6 <---
model = GATClassifier(num_node_features=6, hidden_channels=64, num_classes=2, heads=4).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0003, weight_decay=1e-3)
criterion = torch.nn.CrossEntropyLoss()

def train():
    model.train()
    total_loss = 0
    correct = 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch, data.edge_attr.squeeze(-1) if data.edge_attr is not None else None)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * data.num_graphs
        pred = out.argmax(dim=1)
        correct += int((pred == data.y).sum())
    return total_loss / len(train_dataset), correct / len(train_dataset)

@torch.no_grad()
def evaluate(loader):
    model.eval()
    correct = 0
    total_loss = 0
    for data in loader:
        data = data.to(device)
        out = model(data.x, data.edge_index, data.batch, data.edge_attr.squeeze(-1) if data.edge_attr is not None else None)
        loss = criterion(out, data.y)
        total_loss += loss.item() * data.num_graphs
        pred = out.argmax(dim=1)
        correct += int((pred == data.y).sum())
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# 7. Run the Training Loop
best_val_acc = 0.0
epochs = 120

print("\nBeginning GPU-Accelerated Model Optimization with 6 Topological Features...")
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train()
    val_loss, val_acc = evaluate(val_loader)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        checkpoint_path = os.path.join(checkpoint_dir, 'best_gat_model.pt')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc
        }, checkpoint_path)
        print(f"Epoch {epoch:03d} | Train Acc: {train_acc:.4f} | New Best Val Acc: {val_acc:.4f} [SAVED]")
    else:
        if epoch % 10 == 0:
            print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

print(f"\nTraining Complete! Top Validation Accuracy: {best_val_acc:.4f}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 987.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.6 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.3.0+cu121.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 100.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 105.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 31.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.6/949.6 kB 62.3 MB/s eta 0:00:00
Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_sparse/_version_cuda.so
  import torch_geometric.typing


DataLoaders Ready. Train Batches: 20 | Val Batches: 5
Using execution device: cpu

Beginning GPU-Accelerated Model Optimization with 6 Topological Features...
Epoch 001 | Train Acc: 0.4943 | New Best Val Acc: 0.4656 [SAVED]
Epoch 002 | Train Acc: 0.4992 | New Best Val Acc: 0.5344 [SAVED]
Epoch 009 | Train Acc: 0.5304 | New Best Val Acc: 0.5420 [SAVED]
Epoch 010 | Train Loss: 2.3323 | Train Acc: 0.5140 | Val Loss: 0.6951 | Val Acc: 0.5344
Epoch 020 | Train Loss: 0.9627 | Train Acc: 0.4860 | Val Loss: 0.6954 | Val Acc: 0.5344
Epoch 030 | Train Loss: 0.9110 | Train Acc: 0.5008 | Val Loss: 0.6931 | Val Acc: 0.5344
Epoch 040 | Train Loss: 0.7735 | Train Acc: 0.4877 | Val Loss: 0.6907 | Val Acc: 0.5344
Epoch 050 | Train Loss: 0.7263 | Train Acc: 0.5074 | Val Loss: 0.6910 | Val Acc: 0.5344
Epoch 060 | Train Loss: 0.7402 | Train Acc: 0.4975 | Val Loss: 0.6922 | Val Acc: 0.5344
Epoch 070 | Train Loss: 0.7610 | Train Acc: 0.4844 | Val Loss: 0.6905 | Val Acc: 0.5344
Epoch 080 | Train Loss: 0.7201